<a href="https://colab.research.google.com/github/aelshehawy/gesis-python-2026/blob/main/Day%203/Exercises/Solutions/Day3_Session6_Exercises_Solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise Solutions: Visualization & Your Masterpiece

## Introduction to Python - GESIS Workshop 2026

**Day 3 · Session 6 (Afternoon)** · Thursday, 27 August 2026

**Ashrakat Elshehawy** (Course Convenor, UCL) · a.elshehawy@ucl.ac.uk

**Victor Kreitmann** (Teaching Assistant, UCL) · victor.kreitmann.24@ucl.ac.uk

---

These are the worked solutions - often there is **more than one correct way** to solve an exercise. If your code produces the right result with a different approach, that is completely fine!

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style="whitegrid")

DATA_URL = "https://raw.githubusercontent.com/aelshehawy/gesis-python-2026/main/data/"
gapminder = pd.read_csv(DATA_URL + "gapminder.csv")
gm2007 = gapminder[gapminder["year"] == 2007]

print("Ready!", gapminder.shape)

## Exercise 1 - groupby warm-up

1. Mean **GDP per capita** by continent, 2007.
2. **Median** population by continent, 2007.
3. For the whole dataset (all years): mean, min, and max life expectancy by continent - in **one** `.agg()` call.
4. (Bonus) Which continent's mean life expectancy **improved most** between 1952 and 2007? (Two groupbys and a subtraction - or one clever pivot if you feel adventurous.)

**Solution:**

In [ ]:
# 1. Mean GDP per capita by continent (2007)
print(gm2007.groupby("continent")["gdpPercap"].mean())

# 2. Median population by continent
print(gm2007.groupby("continent")["pop"].median())

In [ ]:
# 3. Three statistics in one shot
gapminder.groupby("continent")["lifeExp"].agg(["mean", "min", "max"])

In [ ]:
# 4. (Bonus) Improvement 1952 → 2007 per continent:
gm1952 = gapminder[gapminder["year"] == 1952]

mean_1952 = gm1952.groupby("continent")["lifeExp"].mean()
mean_2007 = gm2007.groupby("continent")["lifeExp"].mean()

improvement = mean_2007 - mean_1952       # pandas aligns the continents automatically!
print(improvement.sort_values(ascending=False))
# Asia gained the most (~24 years) - the great convergence story of the 20th century.

## Exercise 2 - Election dates

`german_elections.csv` in the course data lists German federal elections and their turnout.

1. Load it and check `.dtypes` - what type is `election_date` when it arrives?
2. Convert it with `pd.to_datetime()`.
3. Create a `year` column using `.dt.year`.
4. In which **month** do Germans usually vote? Extract the month and use `.value_counts()`. One election breaks the pattern - which one?
5. Sort by turnout: which election had the highest, which the lowest?
6. (Bonus) Plot turnout over the years as a labeled line chart with markers. What is the story of the last two elections?

In [ ]:
# The code you were given:
elections = pd.read_csv(DATA_URL + "german_elections.csv")

**Solution:**

In [ ]:
elections = pd.read_csv(DATA_URL + "german_elections.csv")

# 1. It arrives as 'object' - plain text:
print(elections.dtypes)

# 2. Convert to real dates:
elections["election_date"] = pd.to_datetime(elections["election_date"])

# 3. Extract the year:
elections["year"] = elections["election_date"].dt.year
elections

In [ ]:
# 4. The voting month: September - with ONE exception, the snap election
#    of February 2025 after the governing coalition collapsed in late 2024.
elections["month"] = elections["election_date"].dt.month
print(elections["month"].value_counts())

In [ ]:
# 5. Turnout extremes:
print(elections.sort_values("turnout_percent", ascending=False)[["year", "turnout_percent"]])
# Highest: 2025 (82.5) just ahead of 1998 (82.2); lowest: 2009 (70.8).

In [ ]:
# 6. (Bonus) The turnout story:
plt.figure(figsize=(8, 4))
plt.plot(elections["year"], elections["turnout_percent"], marker="o")
plt.title("Turnout in German federal elections, 1998-2025")
plt.xlabel("Election year")
plt.ylabel("Turnout (%)")
plt.show()
# Turnout slid to its 2009 low, recovered ever since, and the polarized
# 2021-2025 period pushed it back up to late-1990s levels.

## Exercise 3 - Your country's story, in lines

1. Pick any country from the data (`gapminder["country"].unique()` shows your options).
2. Plot its **population** over time as a labeled line chart (title, axis labels!). Add markers (`marker="o"`).
3. Make a second line chart of its **GDP per capita** over time, in a different color.
4. (Bonus) Put both charts side by side with `plt.subplots(1, 2, figsize=(12, 4))`.

**Solution:**

In [ ]:
# Any country works - we take Egypt. Change one variable to switch!
my_country = "Egypt"
subset = gapminder[gapminder["country"] == my_country]

# 2. Population over time
plt.figure(figsize=(8, 4))
plt.plot(subset["year"], subset["pop"] / 1_000_000, marker="o")
plt.title(f"Population of {my_country}, 1952-2007")
plt.xlabel("Year")
plt.ylabel("Population (millions)")
plt.show()

In [ ]:
# 3. GDP per capita, different color
plt.figure(figsize=(8, 4))
plt.plot(subset["year"], subset["gdpPercap"], marker="o", color="tab:red")
plt.title(f"GDP per capita of {my_country}, 1952-2007")
plt.xlabel("Year")
plt.ylabel("GDP per capita (dollars)")
plt.show()

In [ ]:
# 4. (Bonus) Side by side: axes is a list of two panels
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(subset["year"], subset["pop"] / 1_000_000, marker="o")
axes[0].set_title(f"{my_country}: population (millions)")

axes[1].plot(subset["year"], subset["gdpPercap"], marker="o", color="tab:red")
axes[1].set_title(f"{my_country}: GDP per capita")

plt.tight_layout()
plt.show()

## Exercise 4 - The comparison chart

1. Choose **three countries** whose comparison interests you.
2. Draw their life expectancy over time in **one** chart - one line each, with a loop (!), labels, and a legend.
3. In a comment: what story does your chart tell in one sentence?

**Solution:**

In [ ]:
# Three countries, one loop - each pass draws one labeled line:
my_three = ["China", "India", "Nigeria"]

plt.figure(figsize=(9, 5))
for country in my_three:
    subset = gapminder[gapminder["country"] == country]
    plt.plot(subset["year"], subset["lifeExp"], marker="o", linewidth=2, label=country)

plt.title("Life expectancy: three rising giants, 1952-2007")
plt.xlabel("Year")
plt.ylabel("Life expectancy (years)")
plt.legend()
plt.show()

# Story in one sentence: China's early lead survived even the visible famine dip
# around 1960, while India and Nigeria climbed more slowly - and a gap remains.

## Exercise 5 - Bars: the population giants

1. Make a **horizontal bar chart** of the 10 most populous countries in 2007 (population in **millions**).
2. Largest at the top! *(You know the invert trick.)*
3. (Bonus) Color the bars by passing a list of colors - make China and India stand out. *(Hint: build the color list with a list comprehension over the country names!)*

**Solution:**

In [ ]:
top_pop = gm2007.sort_values("pop", ascending=False).head(10).copy()
top_pop["pop_millions"] = top_pop["pop"] / 1_000_000

# 3. (Bonus) A color per bar via list comprehension - highlight the two giants:
colors = ["tab:red" if c in ["China", "India"] else "tab:gray" for c in top_pop["country"]]

plt.figure(figsize=(9, 5))
plt.barh(top_pop["country"], top_pop["pop_millions"], color=colors)
plt.gca().invert_yaxis()                       # biggest on top
plt.title("The population giants of 2007")
plt.xlabel("Population (millions)")
plt.show()

## Exercise 6 - Distributions

1. Histogram of `gdpPercap` in 2007, 25 bins. Ugly, right? Why? *(Comment!)*
2. Now histogram `np.log10(gm2007["gdpPercap"])` - better! What does a value of 4 on this axis mean in dollars?
3. (Bonus) Instead of the log trick, try seaborn's `sns.histplot(..., log_scale=True)` - prettiest of all.

**Solution:**

In [ ]:
# 1. Raw GDP histogram: almost everything squashed into the first bins,
#    because a few very rich countries stretch the axis - a heavily SKEWED variable.
plt.figure(figsize=(8, 4))
plt.hist(gm2007["gdpPercap"], bins=25, edgecolor="white")
plt.title("GDP per capita, 2007 (raw) - skew alert!")
plt.xlabel("GDP per capita (dollars)")
plt.ylabel("Number of countries")
plt.show()

In [ ]:
# 2. Log10 version: now the shape is visible.
#    A value of 4 on this axis means 10^4 = 10,000 dollars.
plt.figure(figsize=(8, 4))
plt.hist(np.log10(gm2007["gdpPercap"]), bins=25, color="tab:orange", edgecolor="white")
plt.title("GDP per capita, 2007 (log10)")
plt.xlabel("log10 of GDP per capita  (3 = $1k, 4 = $10k, 5 = $100k)")
plt.ylabel("Number of countries")
plt.show()

In [ ]:
# 3. (Bonus) seaborn does the log axis natively - with real dollar labels:
plt.figure(figsize=(8, 4))
sns.histplot(data=gm2007, x="gdpPercap", bins=25, log_scale=True)
plt.title("GDP per capita, 2007 (seaborn, log axis)")
plt.show()

## Exercise 7 - The full seaborn scatter

Recreate the session's showpiece scatter **for the year 1957** (not 2007!):

- x = GDP per capita (log scale), y = life expectancy
- color by continent, size by population
- proper title and axis labels

Compare it (by eye) with the 2007 version from the session: what changed in 50 years?

**Solution:**

In [ ]:
gm1957 = gapminder[gapminder["year"] == 1957]

plt.figure(figsize=(10, 6))
sns.scatterplot(data=gm1957, x="gdpPercap", y="lifeExp",
                hue="continent", size="pop", sizes=(20, 800), alpha=0.7)

plt.xscale("log")
plt.title("Wealth & health in 1957 - a much poorer, much sicker world")
plt.xlabel("GDP per capita (dollars, log scale)")
plt.ylabel("Life expectancy (years)")
plt.legend(bbox_to_anchor=(1.02, 1))
plt.show()

# Compared with 2007: the whole cloud sits lower and further left -
# and Asia's bubbles have not yet made their great journey to the top right.

## Exercise 8 - Boxes & groups

1. Boxplot of **GDP per capita** by continent, 2007. One continent's box is barely visible - why? Fix it by making the y-axis logarithmic (`plt.yscale("log")`).
2. (Bonus) Try `sns.violinplot` with the same data - what extra information does it show over the boxplot?

**Solution:**

In [ ]:
# 1. Oceania has only 2 countries (Australia & New Zealand) with near-identical
#    values - its box collapses to a sliver. And GDP's skew squashes everything else.
plt.figure(figsize=(9, 5))
sns.boxplot(data=gm2007, x="continent", y="gdpPercap", hue="continent", palette="Set2")
plt.yscale("log")                     # the fix: log axis
plt.title("GDP per capita by continent, 2007 (log scale)")
plt.xlabel("")
plt.show()

In [ ]:
# 2. (Bonus) Violin = boxplot + the full SHAPE of the distribution:
#    you can see e.g. Asia's wide spread and Europe's rich-country bulge.
plt.figure(figsize=(9, 5))
sns.violinplot(data=gm2007, x="continent", y="gdpPercap", hue="continent", palette="Set2")
plt.yscale("log")
plt.title("Same data, violin edition")
plt.xlabel("")
plt.show()

## Exercise 9 - Animate something!

Your turn with plotly express:

1. Build an **animated bar chart race**: `px.bar` with `x="pop"`, `y="country"`, `animation_frame="year"`, `orientation="h"` - using only the **6 most populous countries of 2007** (filter the full data to those 6 countries first with `.isin()`!). Add `range_x=[0, 1.4e9]` so the axis stays fixed.
2. **Or** build an animated choropleth of `gdpPercap` over time (session code is your template - try `color_continuous_scale="Plasma"` and, since GDP is skewed, map `np.log10` of it or set `range_color`).
3. Press play and enjoy what you just made from scratch.

**Solution:**

In [ ]:
# 1. The bar chart race:
top6_names = gm2007.sort_values("pop", ascending=False).head(6)["country"]
race_data = gapminder[gapminder["country"].isin(top6_names)]

fig = px.bar(
    race_data,
    x="pop", y="country",
    orientation="h",                 # horizontal bars
    animation_frame="year",          #
    range_x=[0, 1.4e9],              # fixed axis so growth is visible
    color="country",
    title="Population race of the giants, 1952-2007 (press Play!)",
)
fig.show()

In [ ]:
# 2. The animated GDP map (log10 for readable colors):
map_data = gapminder.copy()
map_data["log10_gdp"] = np.log10(map_data["gdpPercap"])

fig = px.choropleth(
    map_data,
    locations="iso_alpha",
    color="log10_gdp",
    hover_name="country",
    hover_data={"gdpPercap": ":,.0f", "log10_gdp": False, "iso_alpha": False},
    animation_frame="year",
    color_continuous_scale="Plasma",
    range_color=[2.5, 5],            # 10^2.5 ≈ $300 → 10^5 = $100,000
    title="GDP per capita (log scale), 1952-2007 (press Play!)",
)
fig.show()

## Exercise 10 - YOUR MASTERPIECE

Time to fly solo. Pick **one research question** - one of these, or (better!) your own:

- *Has the wealth-health relationship gotten stronger or weaker over time?*
- *Which African countries defied the continent's average trend - and when?*
- *Population growth: which continent will the 21st century belong to?*
- *Find the biggest economic "catch-up story" in the data (hint: gdpPercap ratios 2007/1952...).*

**Your deliverable** (template below):

1. A markdown cell stating your **question**.
2. **Data work**: filter/groupby/new columns - whatever your question needs.
3. **At least two visualizations**, properly labeled - at least one should be *presentation-ready* (the kind you'd put in a paper or talk).
4. A **one-sentence conclusion** in markdown.
5. Post a screenshot of your favorite chart in the Zoom chat - we'll do a mini gallery in the last 15 minutes!

**Solution:**

**Question: *Find the biggest economic catch-up story in the data.***

*(Yours will look different - that is the point! This shows one possible full arc.)*

In [ ]:
# --- Step 1: data work ---
# Compare each country's gdpPercap in 2007 vs 1952 → growth ratio.
gm1952 = gapminder[gapminder["year"] == 1952][["country", "continent", "gdpPercap"]]
gm2007x = gapminder[gapminder["year"] == 2007][["country", "gdpPercap", "lifeExp"]]

# Merge the two years side by side (suffixes label the columns):
compare = gm1952.merge(gm2007x, on="country", suffixes=("_1952", "_2007"))

# The growth ratio as a new column:
compare["growth_ratio"] = compare["gdpPercap_2007"] / compare["gdpPercap_1952"]

# The top catch-up stories:
top_growth = compare.sort_values("growth_ratio", ascending=False).head(10)
top_growth[["country", "gdpPercap_1952", "gdpPercap_2007", "growth_ratio"]].round(1)

In [ ]:
# --- Step 2: visualization 1 - the top-10 catch-up ranking ---
plt.figure(figsize=(9, 5))
colors = ["tab:red" if c == "Korea, Rep." else "tab:blue" for c in top_growth["country"]]
plt.barh(top_growth["country"], top_growth["growth_ratio"], color=colors)
plt.gca().invert_yaxis()
plt.title("Economic catch-up champions: GDP per capita, 2007 vs 1952 (×)")
plt.xlabel("2007 GDP per capita as a multiple of 1952")
plt.show()

In [ ]:
# --- Step 3: visualization 2 - South Korea's path, in context ---
story = ["Korea, Rep.", "Germany", "Egypt"]

plt.figure(figsize=(9, 5))
for country in story:
    subset = gapminder[gapminder["country"] == country]
    plt.plot(subset["year"], subset["gdpPercap"], marker="o", linewidth=2, label=country)

plt.yscale("log")
plt.title("The South Korean miracle (log scale)")
plt.xlabel("Year")
plt.ylabel("GDP per capita (dollars, log)")
plt.legend()
plt.show()

### Conclusion

**South Korea grew its GDP per capita more than 22-fold between 1952 and 2007 - the steepest sustained catch-up in the dataset - rising from Egypt's level to Germany's neighborhood within two generations.**

---
**Congratulations on finishing the course - now go analyze your own data!**